In [1]:
import time
from labjack import ljm

# ==========================================
# CONFIGURATION & CALIBRATION CONSTANTS
# ==========================================
INPUT_REGISTER = "AIN4"  # LabJack Register for AIO4

# Flow Rate (Q) Limits in GPM
Q_MIN = 0.79
Q_MAX = 79.00

# LJTCS Specific Voltage Limits (From your documentation)
V_MIN = 0.472  # Voltage at 4mA
V_MAX = 2.360  # Voltage at 20mA


def voltage_to_gpm(voltage):
    """Converts LJTCS amplified voltage to mA and GPM based on documentation."""
    # 1. Official LJTCS Equation: mA = 8.475 * Volts
    current_mA = 8.475 * voltage

    # 2. Linear Interpolation (Mapping Voltage directly to GPM)
    # Formula: Q = Q_MIN + (V - V_MIN) * (Q_MAX - Q_MIN) / (V_MAX - V_MIN)
    if voltage < V_MIN:
        # Handle zero-flow / minor low-end noise safely
        gpm = Q_MIN
    else:
        gpm = Q_MIN + (voltage - V_MIN) * (Q_MAX - Q_MIN) / (V_MAX - V_MIN)

    return current_mA, gpm


def main():
    try:
        handle = ljm.openS("T7", "ANY", "ANY")
        info = ljm.getHandleInfo(handle)
        print(f"Connected to LabJack T7 [Serial: {info[2]}]")
        print("-" * 50)
        print(f"{'Voltage (V)':<15}{'Current (mA)':<15}{'Flow Rate (GPM)':<15}")
        print("-" * 50)

        # Ensure single-ended reading relative to GND
        ljm.eWriteName(handle, f"{INPUT_REGISTER}_NEGATIVE_CH", 199)

        # Set T7 Range to +/- 10.0V.
        # Note: Do NOT use 1.0V range here, because the LJTCS outputs up to 2.36V,
        # which would clip a 1.0V max analog input scale.
        ljm.eWriteName(handle, f"{INPUT_REGISTER}_RANGE", 10.0)

        while True:
            # Read voltage from AIO4/AIN4
            voltage = ljm.eReadName(handle, INPUT_REGISTER)

            # Perform corrected conversions
            current_mA, flow_gpm = voltage_to_gpm(voltage)

            # Print data formatted to 4 decimal places
            print(
                f"{voltage:<15.4f}{current_mA:<15.4f}{flow_gpm:<15.4f}",
                end="\r",
            )

            time.sleep(0.5)

    except ljm.LJMError as e:
        print(f"\nLabJack LJM Error: {e}")
    except KeyboardInterrupt:
        print("\nTesting stopped by user.")
    finally:
        if "handle" in locals():
            ljm.close(handle)
            print("LabJack connection closed.")


if __name__ == "__main__":
    main()

Connected to LabJack T7 [Serial: 470042305]
--------------------------------------------------
Voltage (V)    Current (mA)   Flow Rate (GPM)
--------------------------------------------------
0.4644         3.9354         0.7900         
Testing stopped by user.
LabJack connection closed.
